# Problem Definition and Dataset

This notebook introduces the Chicago Taxi Trips dataset and defines the machine learning problem.

The objectives are to:

- explore the dataset structure
- verify Big Data requirements
- inspect the schema
- validate data quality before preprocessing

# Project Configuration

In [ ]:
import os, subprocess, json


BASE = "/content/drive/MyDrive/7006SCN"
DATA = f"{BASE}/data"
PROC = f"{BASE}/processed"
MODELS = f"{BASE}/models"
META = f"{BASE}/metadata"
OUTPUTS = f"{BASE}/outputs"

# ------------------------------------------------------------------
# Artefact paths — referenced by SAVE (producer) and LOAD (consumer)
# ------------------------------------------------------------------
RAW_DATA_PATH = f"{DATA}/taxi.csv"

PROC_TRAIN = f"{PROC}/training.parquet"
PROC_TEST = f"{PROC}/test.parquet"

PIPELINE_PATH = f"{MODELS}/preprocessing_pipeline"

LR_MODEL_PATH = f"{MODELS}/lr_model"
RF_MODEL_PATH = f"{MODELS}/rf_model"
GBT_MODEL_PATH = f"{MODELS}/gbt_model"

TASK1_META = f"{META}/task1_metadata.json"
TASK2_META = f"{META}/task2_metadata.json"
TASK3_META = f"{META}/task3_metadata.json"


for p in [DATA, PROC, MODELS, META, OUTPUTS]:
    os.makedirs(p, exist_ok=True)

def verify_exists(path, label=""):
"""subprocess verify — prints ls -lh for the path"""
if result.returncode == 0:
print(f"✓ {label or path}:")
print(result.stdout.strip()) else:
raise FileNotFoundError(
f"NOT FOUND: {path}\n{result.stderr}")
print("Shared constants loaded ✓")

## Installing PySpark

In [ ]:
# ----------------------------------
# Install pyspark
# ----------------------------------

!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,055 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,611 kB]
Get:14

# Initialising Spark session

In [ ]:
# ----------------------------------
# Initialising new spark session
# ----------------------------------

from pyspark.sql import SparkSession
import time

spark = (
    SparkSession.builder
    .appName("7006SCN_ME_16882940_Task_1")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Session started:", spark.sparkContext.applicationId)


Spark version: 4.0.2
Session started: local-1781269295685


# 1 . Data Ingestion — Task 1 (Dataset loading, schema inspection, file size)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ----------------------------------
# Loading csv into spark data frame:
# ----------------------------------

taxi_df = spark.read.csv(
    RAW_DATA_PATH,
    header=True,
    inferSchema=False
)

# ----------------------------------
# Data ingestion
# ----------------------------------

t_ingest_start = time.time()

ingest_time = time.time() - t_ingest_start
print(f"Ingestion time : {ingest_time:.2f}s")
print(f"Row count      : {taxi_df.count():,}")
print(f"Column count   : {len(taxi_df.columns)}")

Ingestion time : 0.00s
Row count      : 12,877,995
Column count   : 23


In [ ]:
# ----------------------------------
# Viewing the data
# ----------------------------------

taxi_df.show(20)

+--------------------+--------------------+--------------------+--------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+------+------+-----+------+----------+------------+--------------------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+--------------------------+
|             Trip ID|             Taxi ID|Trip Start Timestamp|  Trip End Timestamp|Trip Seconds|Trip Miles|Pickup Census Tract|Dropoff Census Tract|Pickup Community Area|Dropoff Community Area|  Fare|  Tips|Tolls|Extras|Trip Total|Payment Type|             Company|Pickup Centroid Latitude|Pickup Centroid Longitude|Pickup Centroid Location|Dropoff Centroid Latitude|Dropoff Centroid Longitude|Dropoff Centroid  Location|
+--------------------+--------------------+--------------------+--------------------+------------+----------+-------------------+--------------------+--

In [ ]:
# ----------------------------------
# Schema inspection
# ----------------------------------

taxi_df.printSchema()

root
 |-- Trip ID: string (nullable = true)
 |-- Taxi ID: string (nullable = true)
 |-- Trip Start Timestamp: string (nullable = true)
 |-- Trip End Timestamp: string (nullable = true)
 |-- Trip Seconds: string (nullable = true)
 |-- Trip Miles: string (nullable = true)
 |-- Pickup Census Tract: string (nullable = true)
 |-- Dropoff Census Tract: string (nullable = true)
 |-- Pickup Community Area: string (nullable = true)
 |-- Dropoff Community Area: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Tips: string (nullable = true)
 |-- Tolls: string (nullable = true)
 |-- Extras: string (nullable = true)
 |-- Trip Total: string (nullable = true)
 |-- Payment Type: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Pickup Centroid Latitude: string (nullable = true)
 |-- Pickup Centroid Longitude: string (nullable = true)
 |-- Pickup Centroid Location: string (nullable = true)
 |-- Dropoff Centroid Latitude: string (nullable = true)
 |-- Dropoff Centroid

## Summary

In this notebook I:

- loaded and inspected the dataset
- completed feature engineering
- prepared the data for distributed machine learning

The processed dataset is now ready for model training.